In [ ]:
#Inputs: The model will take in scenario descriptors, specifically demand changes that follow road closures or transit maintenance.
#Outputs: The model needs to predict SUMO-level performance metrics.
#Specifically, it must output an updated time-of-day timing plan.
#Outputs: It must also output a scenario impact summary covering delay, throughput, and queues based on SUMO evaluation outputs.

#Use supervised regression

#Training Data:
#Training follows a curriculum approach where the trained ASCE controller is
#evaluated across progressively more complex demand and infrastructure
#perturbations; PIRA is then trained on those resulting scenario-outcome pairs.

In [ ]:
import xml.etree.ElementTree as ET
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import Linear

In [ ]:
def parse_sumo_network(xml_file_path):
    """
    Parses a SUMO network XML file to create a PyTorch Geometric edge_index.
    XML file should contain a list of junctions as nodes and roads as edges.
    """
    #Load & parse the XML file
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    #Map string junction IDs to integer indices (PyG requirement apparently)
    node_id_to_idx = {}
    current_idx = 0

    #Find all junctions to register as nodes
    for junction in root.findall('junction'):
        j_id = junction.get('id')
        j_type = junction.get('type')

        #Only map valid junctions. (Filtering also possible by j_type == 'traffic_light' if you only want signalized intersections in the graph)
        if j_id and j_id not in node_id_to_idx:
            node_id_to_idx[j_id] = current_idx
            current_idx += 1

    #Extract edges (roads connecting the junctions)
    source_nodes = []
    target_nodes = []

    for edge in root.findall('edge'):
        from_node = edge.get('from')
        to_node = edge.get('to')

        #Only keep edges that connect known junctions (ignores internal/dummy edges)
        if from_node in node_id_to_idx and to_node in node_id_to_idx:
            source_nodes.append(node_id_to_idx[from_node])
            target_nodes.append(node_id_to_idx[to_node])

    #Create the PyTorch Geometric edge_index tensor
    edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

    return edge_index, node_id_to_idx

#--- Usage ---
#Upload 'osm.net.xml' to Colab first
#edge_index, node_mapping = parse_sumo_network('osm.net.xml')
#edge_index: 2D tensor, [2, num_edges]
#print(f"Created graph with {len(node_mapping)} nodes and {edge_index.shape[1]} edges.")
#print(edge_index)

In [ ]:
class PIRAModel(torch.nn.Module):
    def __init__(self, num_node_features, num_output_targets):
        super(PIRAModel, self).__init__()

        # 1. First Message Passing Layer
        # Takes in your raw traffic data (e.g., queues, speeds) and creates hidden features
        self.conv1 = GCNConv(num_node_features, 64)

        # 2. Second Message Passing Layer
        # Intersections talk to their neighbors again to get a wider view of the corridor
        self.conv2 = GCNConv(64, 32)

        # 3. Output Layer (Regression)
        # Maps the final graph features to your target metrics (delay, throughput, queues)
        self.out = Linear(32, num_output_targets)

    def forward(self, data):
        # Unpack the graph data object
        x, edge_index = data.x, data.edge_index

        # Pass data through the first graph layer and apply a ReLU activation (keeps numbers positive)
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # Pass through the second graph layer
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Pass through the final linear layer to get your numerical predictions
        # No activation here because regression targets (like delay) can have any range
        predictions = self.out(x)

        return predictions

#Usage
#This assumes we have 6 input features per intersection (e.g., queues, speeds). Check with Yash later.
#Outputs three things we want to predict (delay, throughput, queue_total)
#pira_net = PIRAModel(num_node_features=6, num_output_targets=3)
#print(pira_net)